# GJ 3929b

GJ 3929b is a exoplanet observed as part of the [Rocky Worlds](https://rockyworlds.stsci.edu/index.html) survey with JWST/MIRI and HST/COS+STIS.  
In this notebook, we load a Rocky Worlds light curve and HST spectrum.  We use lcviz to bin the data and overlay an eclipse model, and jdaviz to model the Ly alpha emission line.  

- Last Updated: November 11, 2026
- lcviz version: 2.0.1.dev21+g978e0fe61
- jdaviz version: 5.1.dev250+g2632c5d9d

In [ ]:
# note: we won't call functions in `lcviz` directly in this notebook, we 
# import lcviz to add light curve features into jdaviz:
import jdaviz as jd
import lcviz

# several warnings for unit issues that are not important, so we hide those
import warnings
import warnings 
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", category=UserWarning, module="jupyter_client")

# Use LCviz to load a lightcurve into jdaviz and bin the data


## Create an instance of the app

Since we are going to use two instances - one for the lightcurve and one for the spectrum - we open the app twice.  We use the ``new_app`` call to initialize the app.

In [ ]:
jd1 = jd.new_app()
jd1.show()


## Load the LightCurve eclipse h5 file into jdaviz

In [ ]:
filename_lc = 'hlsp_rocky-worlds_jwst_miri_gj3929b-ecl003_f1500w_v1.0.1_lc.h5'
jd1.load(filename_lc, format='Light Curve')


## Change the data to show the cleaned Flux instead of the raw


In [ ]:
fl = jd1.plugins['Flux Column']
fl.open_in_tray() #optional, to see the plugin in the UI

fl.dataset = 'GJ 3929 b'
fl.flux_column = 'cleanedFlux'


## Bin the cleaned Flux

In [ ]:
bi = jd.plugins['Binning']
bi.open_in_tray() #optional, to see the plugin in the UI
 
bi.dataset = 'GJ 3929 b'
bi.n_bins = 30
bi.add_results.label = 'binned GJ 3929 b'
bi.bin()


## Change the plot syle

In [ ]:
po = jd1.plugins['Plot Options']
po.open_in_tray() #optional, to see the plugin in the UI

po.viewer = 'flux-vs-time'
po.layer = 'binned GJ 3929 b'
po.marker_size_scale = 10
po.marker_size = 10
po.marker_color = '#FF0000'


## Load in the eclipse model to compare

In [ ]:
jd1.load(filename_lc, format='Light Curve', extension = 'astroModel')


---

# Use Jdaviz to load in the HST spectrum and fit the Lyα emission line

## Create a new app 

In [ ]:
jd2 = jd.new_app()
jd2.show()

## Load in the HST COS/STIS spectrum

In [ ]:
jd2.load('hlsp_rocky-worlds_hst_cos-stis_gj3929_multi_v1.0_native-spec.fits', format='1D Spectrum')


## Create a region using specutils and import it into the app.

We use ``rename_subset`` to change the name from ``Subset XX`` to ``Lya``

In [ ]:

from specutils.spectra import SpectralRegion
from astropy import units as u
sub = jd2.plugins['Subset Tools']

sr = SpectralRegion(1214.5 * u.AA, 1217 * u.AA)
sub.import_region(sr, combination_mode='new')
sub.rename_subset('Subset 1', 'Lya')

## Change the zoom limits

In [ ]:
viewer = jd2.viewers['1D Spectrum']
viewer.set_limits(x_min=1200, x_max=1230, y_max=2.5e-14)

## Use the model fitting plugin to fit a Voigt profile to the Lyα emission line

In [ ]:
model_fit = jd2.plugins['Model Fitting']
model_fit.open_in_tray() #optional, to see the plugin in the UI

model_fit.spectral_subset = 'Lya'

model_fit.model_component = 'Const1D'
model_fit.model_component_label = 'C'
model_fit.create_model_component()

model_fit.model_component = 'Voigt1D'
model_fit.model_component_label = 'V'
model_fit.create_model_component()

model_fit.calculate_fit()
model_fit.export_table()

## Change the plot style to see the fitted line

In [ ]:
po = jd2.plugins['Plot Options']
po.open_in_tray() #optional, to see the plugin in the UI

po.layer = 'Lya'
po.line_visible = False
po.line_color = '#B41F1F'

po.layer = 'model'
po.line_width = 3


---

written by: Sarah Betti, Nov 2026